# Cancelación condicional de dilataciones en la corrección de Schur

Este cuaderno ejecuta el lema abstracto para el patrón adyacente $W(b_{21})V_\gamma\operatorname{Op}_M(r_{11})V_{\gamma^{-1}}W(b_{12})$. No identifica ese patrón con los bloques actuales del artículo: esa aplicación permanece `BLOCKED_WITH_PRECISE_OBLIGATION`.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

import sympy as sp  # noqa: E402
from symbolic_operator_calculus import (  # noqa: E402
    AlgebraMembershipStatus, AssumptionContext, CancellationLayerStatus,
    ComplexDomain, DerivationRelationKind, ExactBlock, FormalRegularizer,
    MellinExpression, MellinPseudodifferentialOperator, MellinSymbol,
    MellinSymbolDependency, MellinSymbolDomain, OperatorAtom,
    RegularizerMellinRepresentation, RegularizerRepresentationStatus,
    SchurBlockRepresentation, SchurCancellationStatus, SingularSet,
    WeightedDilationOperator, WeightedLpSpace, WienerHopfOperator,
    cancel_relative_dilations_in_schur_correction, mellin_frequency,
    output_spatial_variable,
)
print('API pública importada; no se materializa ningún kernel del regularizador.')

API pública importada; no se materializa ningún kernel del regularizador.


## 1. Espacio ponderado y dilataciones

La norma es $\|f\|^p=\int_0^\infty |x^\delta f(x)|^p\,dx$ y la normalización isométrica es $U_\gamma f(x)=\gamma^{\delta+1/p}f(\gamma x)$.

In [2]:
gamma = sp.Symbol('gamma', positive=True)
delta = sp.Symbol('delta', real=True)
lam = sp.Symbol('lambda', real=True)
x = sp.Symbol('x', positive=True)
context = AssumptionContext((sp.StrictGreaterThan(gamma, 0),))
space = WeightedLpSpace(
    label='L^2_delta(R_+)', p=sp.Integer(2), weight_exponent=delta,
    assumptions=context, source='article weighted norm',
)
V_gamma = WeightedDilationOperator(
    atom=OperatorAtom('V_gamma', kind='weighted_dilation'), gamma=gamma,
    domain=space, codomain=space, assumptions=context,
    source='U_gamma f(x)=gamma^(delta+1/p) f(gamma*x)',
    evidence=('direct weighted-norm change of variables',),
)
V_gamma_inverse = V_gamma.inverse(
    OperatorAtom('V_gamma_inverse', kind='weighted_dilation')
)
assert V_gamma.is_isometric_normalization
assert sp.simplify(V_gamma.gamma * V_gamma_inverse.gamma) == 1
print('space:', space.label)
print('normalization exponent:', space.normalization_exponent)
print('inverse scale:', V_gamma_inverse.gamma)

space: L^2_delta(R_+)
normalization exponent: delta + 1/2
inverse scale: 1/gamma


## 2. Modelos de $A_{21}$ y $A_{12}$

Para demostrar la propagación de la relación más débil, ambos modelos se suministran explícitamente módulo compactos. No se reordenan factores ni se atraviesan localizadores.

In [3]:
b21 = 1 + sp.exp(-lam**2)
b12 = 2 + sp.exp(-2 * lam**2)
W21 = WienerHopfOperator(
    atom=OperatorAtom('W_b21', kind='wiener_hopf'), symbol=b21,
    frequency_variable=lam, domain=space, codomain=space,
    symbol_class='declared WH scale-stable class', source='abstract lemma input',
    frequency_scaling_stable=True,
    stability_evidence=('constant positive frequency scaling preserves the declared class',),
)
W12 = WienerHopfOperator(
    atom=OperatorAtom('W_b12', kind='wiener_hopf'), symbol=b12,
    frequency_variable=lam, domain=space, codomain=space,
    symbol_class='declared WH scale-stable class', source='abstract lemma input',
    frequency_scaling_stable=True,
    stability_evidence=('constant positive frequency scaling preserves the declared class',),
)
A21_atom = OperatorAtom('A21')
A12_atom = OperatorAtom('A12')
A21 = SchurBlockRepresentation(
    exact_block=ExactBlock('A', 2, 1), atom=A21_atom, factors=(W21, V_gamma),
    relation_kind=DerivationRelationKind.MOD_COMPACT_EQUIVALENCE,
    source='explicit hypothesis A21 ~= W(b21)V_gamma',
    hypotheses=('bounded factors on the common weighted space',),
    evidence=('supplied compact remainder K21',),
)
A12 = SchurBlockRepresentation(
    exact_block=ExactBlock('A', 1, 2), atom=A12_atom, factors=(V_gamma_inverse, W12),
    relation_kind=DerivationRelationKind.MOD_COMPACT_EQUIVALENCE,
    source='explicit hypothesis A12 ~= V_gamma_inverse W(b12)',
    hypotheses=('bounded factors on the common weighted space',),
    evidence=('supplied compact remainder K12',),
)
print('A21 model:', A21.model)
print('A12 model:', A12.model)

A21 model: Product(factors=(OperatorAtom(name='W_b21', kind='wiener_hopf'), OperatorAtom(name='V_gamma', kind='weighted_dilation')))
A12 model: Product(factors=(OperatorAtom(name='V_gamma_inverse', kind='weighted_dilation'), OperatorAtom(name='W_b12', kind='wiener_hopf')))


## 3. Regularizador abstracto y representación Mellin condicional

$R_{11}$ continúa siendo un átomo abstracto. El objeto separado que lo representa por $\operatorname{Op}_M(r_{11})$ lleva estado condicional y una hipótesis visible.

In [4]:
frequency = mellin_frequency(lam)
radial = output_spatial_variable(x)
mellin_domain = MellinSymbolDomain(
    frequency=frequency, complex_domain=ComplexDomain.real_line(lam),
    spatial_variables=(radial,), assumption_context=context,
    singular_set=SingularSet(), description='radial Mellin-PDO symbol domain',
)
r11_expression = 1 + x * sp.exp(-lam**2)
r11 = MellinSymbol(
    MellinExpression(
        r11_expression, mellin_domain, variables=(frequency, radial),
        evidence=('declared model symbol',), description='conditional r11',
    ),
    MellinSymbolDependency.SPACE_FREQUENCY, description='conditional r11',
)
Op_r11 = MellinPseudodifferentialOperator(
    atom=OperatorAtom('OpM_r11', kind='mellin_pdo'), symbol=r11,
    domain=space, codomain=space, symbol_class='declared radially scale-stable class',
    source='conditional regularizer representation', radial_scaling_stable=True,
    stability_evidence=('direct check from the declared symbol-class definition',),
)
R11 = FormalRegularizer(
    target=ExactBlock('A', 1, 1),
    operator=OperatorAtom('R11', kind='formal_regularizer'),
)
R11_as_mellin = RegularizerMellinRepresentation(
    regularizer=R11, mellin_operator=Op_r11,
    status=RegularizerRepresentationStatus.ASSUMED,
    hypotheses=('R11 equals Op_M(r11) on the stated weighted space',),
    source='conditional lemma hypothesis; not an article-level theorem', space=space,
)
print('abstract regularizer:', R11.operator)
print('representation status:', R11_as_mellin.status.value)

abstract regularizer: OperatorAtom(name='R11', kind='formal_regularizer')
representation status: CONDITIONAL_ON_REGULARIZER_REPRESENTATION


## 4. Término inicial, transformación y término final

La única cancelación exacta interna es $V_\gamma\operatorname{Op}_M(r_{11})V_{\gamma^{-1}}=\operatorname{Op}_M(r_{11}(\gamma x,\lambda))$. Las relaciones de bloque siguen siendo módulo compactos y la representación del regularizador sigue siendo condicional.

In [5]:
result = cancel_relative_dilations_in_schur_correction(
    left_block=A21, regularizer=R11_as_mellin, right_block=A12,
    assumptions=context,
)
assert result.status is SchurCancellationStatus.CONDITIONAL_ON_REGULARIZER_REPRESENTATION
assert result.relation is DerivationRelationKind.MOD_COMPACT_EQUIVALENCE
assert result.classification.dilation_cancellation is CancellationLayerStatus.EXACT_IDENTITY
assert result.classification.initial_block_representations is CancellationLayerStatus.EQUALITY_MODULO_COMPACTS
assert result.classification.regularizer_mellin_representation is CancellationLayerStatus.CONDITIONAL_ON_REGULARIZER_REPRESENTATION
assert result.classification.mixed_algebra_membership is CancellationLayerStatus.UNPROVED_MIXED_ALGEBRA_MEMBERSHIP
assert result.algebra_membership.status is AlgebraMembershipStatus.UNPROVED
assert result.scaled_regularizer is not None
assert sp.simplify(
    result.scaled_regularizer.symbol.as_expr() - r11_expression.subs(x, gamma*x)
) == 0
print('initial term:', result.original_expression)
print('five-factor model:', result.model_expression)
print('final term:', result.expression)
print('radially scaled symbol:', result.scaled_regularizer.symbol.as_expr())
print('status:', result.status.value)
print('relation:', result.relation.value)

initial term: Product(factors=(OperatorAtom(name='A21', kind='operator'), OperatorAtom(name='R11', kind='formal_regularizer'), OperatorAtom(name='A12', kind='operator')))
five-factor model: Product(factors=(OperatorAtom(name='W_b21', kind='wiener_hopf'), OperatorAtom(name='V_gamma', kind='weighted_dilation'), OperatorAtom(name='OpM_r11', kind='mellin_pdo'), OperatorAtom(name='V_gamma_inverse', kind='weighted_dilation'), OperatorAtom(name='W_b12', kind='wiener_hopf')))
final term: Product(factors=(OperatorAtom(name='W_b21', kind='wiener_hopf'), OperatorAtom(name='OpM_r11_radially_scaled', kind='mellin_pdo'), OperatorAtom(name='W_b12', kind='wiener_hopf')))
radially scaled symbol: gamma*x*exp(-lambda**2) + 1
status: CONDITIONAL_ON_REGULARIZER_REPRESENTATION
relation: MOD_COMPACT_EQUIVALENCE


## 5. Obligaciones y salida LaTeX

La transformación no infiere pertenencia a un álgebra Fredholm ni construye el símbolo de la corrección de Schur.

In [6]:
print('proof obligations:')
for obligation in result.proof_obligations:
    print(f'- {obligation.key}: {obligation.statement}')
print('LaTeX:', result.latex)

proof obligations:
- regularizer_mellin_representation: Prove the assumed R11 Mellin-PDO representation in the article's weighted operator space.
- wh_mellin_wh_sandwich_membership: Prove that the final WH--Mellin-PDO--WH sandwich belongs to a closed operator algebra.
- fredholm_algebra_membership: Prove membership in an algebra carrying the required Fredholm calculus.
- schur_correction_symbol: Construct the symbol of the Schur correction only after algebra membership.
LaTeX: W\!\left(1 + e^{- \lambda^{2}}\right)\,\operatorname{Op}_M\!\left(\gamma x e^{- \lambda^{2}} + 1\right)\,W\!\left(2 + e^{- 2 \lambda^{2}}\right)


## 6. Frontera con el artículo

El ejemplo anterior verifica la API bajo hipótesis suministradas. Los bloques reales del artículo contienen un orden distinto, factores de Cauchy/coefficient y localizadores $\chi_\infty$; además, el regularizador completo no está demostrado como un único Mellin PDO. Por ello la conclusión para el artículo es **`BLOCKED_WITH_PRECISE_OBLIGATION`**, no una certificación del pivote.